# Assemble opportunities
Zehui Yin

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
TARGET_CRS = "EPSG:4326"
WORKING_CRS = "EPSG:26917"
HOSPITAL_FILES = {
    "Hamilton": DATA_DIR / "hamilton_hospitals.geojson",
    "Toronto": DATA_DIR / "toronto_hospitals.geojson",
}

## Load hospital opportunity layers

Read both city hospital layers and prepare them for a single downstream opportunity dataset.

In [ ]:
def convert_geometries_to_points(geometry_series):
    area_mask = geometry_series.geom_type.isin(["Polygon", "MultiPolygon"])
    linear_mask = geometry_series.geom_type.isin(["LineString", "MultiLineString"])
    multipoint_mask = geometry_series.geom_type.eq("MultiPoint")

    point_geometries = geometry_series.copy()
    point_geometries.loc[area_mask] = point_geometries.loc[area_mask].representative_point()
    point_geometries.loc[linear_mask | multipoint_mask] = point_geometries.loc[linear_mask | multipoint_mask].centroid

    unsupported_geometry_types = sorted(
        set(point_geometries.geom_type.unique()) - {"Point"}
    )
    if unsupported_geometry_types:
        raise ValueError(
            f"Unsupported geometry types after conversion: {unsupported_geometry_types}"
        )

    return point_geometries


def load_hospital_opportunities(city_name, file_path):
    hospitals = gpd.read_file(file_path).copy()

    if hospitals.crs is None:
        hospitals = hospitals.set_crs(TARGET_CRS)

    hospitals = hospitals.to_crs(WORKING_CRS)
    hospitals["source_city"] = city_name
    hospitals["source_file"] = file_path.name
    hospitals["original_geometry_type"] = hospitals.geometry.geom_type
    hospitals["geometry"] = convert_geometries_to_points(hospitals.geometry)
    hospitals = hospitals.to_crs(TARGET_CRS)

    hospitals["opportunity_id"] = [
        f"{city_name.lower()}_{index + 1:03d}" for index in range(len(hospitals))
    ]
    hospitals["opportunity_name"] = (
        hospitals.get("name", pd.Series(index=hospitals.index, dtype="object"))
        .fillna(hospitals.get("operator", pd.Series(index=hospitals.index, dtype="object")))
        .fillna(hospitals["opportunity_id"])
    )
    hospitals["opportunity_category"] = (
        hospitals.get("healthcare", pd.Series(index=hospitals.index, dtype="object"))
        .fillna(hospitals.get("amenity", pd.Series(index=hospitals.index, dtype="object")))
        .fillna("hospital")
    )

    return hospitals

In [ ]:
hospital_layers = [
    load_hospital_opportunities(city_name, file_path)
    for city_name, file_path in HOSPITAL_FILES.items()
    if file_path.exists()
    ]

hospital_opportunities = gpd.GeoDataFrame(
    pd.concat(hospital_layers, ignore_index=True),
    crs=TARGET_CRS,
 )

hospital_opportunities = hospital_opportunities[[
    "opportunity_id",
    "opportunity_name",
    "opportunity_category",
    "source_city",
    "source_file",
    "original_geometry_type",
    "name",
    "operator",
    "amenity",
    "healthcare",
    "geometry",
]]

print(f"Merged hospital opportunities: {len(hospital_opportunities):,}")
hospital_opportunities.head()

## Validate and export opportunities

Confirm that every feature is now represented as a point and save the merged layer for downstream accessibility analysis.

In [ ]:
geometry_counts = hospital_opportunities.geometry.geom_type.value_counts().rename_axis("geometry_type")
source_counts = hospital_opportunities.groupby("source_city").size().rename("feature_count")

display(geometry_counts.to_frame())
display(source_counts.to_frame())

In [ ]:
hospital_opportunities.explore()

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
output_path = OUTPUT_DIR / "hospital_opportunities.geojson"

hospital_opportunities.to_file(output_path, driver="GeoJSON")

print(f"Saved opportunities to: {output_path}")
print(f"Total features: {len(hospital_opportunities):,}")
print(f"CRS: {hospital_opportunities.crs}")